# Evaluación externa de explicaciones de redes convolucionales mediante perturbaciones pulmonares en radiografías con hallazgos compatibles con tuberculosis

**Extensión retrospectiva con evaluación externa.** Se entrenan DenseNet121 y ResNet50 en Shenzhen y se evalúan en Montgomery. El contraste principal compara, por radiografía anotada, la pérdida de puntuación positiva tras desenfocar una caja índice con la mediana de regiones sin anotación de igual forma digital en el mismo pulmón. Las etiquetas indican hallazgos radiográficos compatibles con TB; no son confirmación microbiológica.

Ejecutar de arriba abajo en Colab con GPU. `QUICK_RUN=1` activa una prueba técnica; sus resultados no se usan como evidencia empírica. La descarga de Kaggle puede requerir autenticación mediante los secretos de Colab o una copia local configurada en `CXR_DATA_ROOT`. Las anotaciones NLM se descargan de su sitio oficial y también pueden cargarse como CSV locales. Los checkpoints existentes solo se reutilizan si su configuración coincide.


## 0. Configuración y reproducibilidad

In [ ]:
# En Colab se instalan dependencias Linux compatibles antes de importar el estudio.
import sys, subprocess
if 'google.colab' in sys.modules:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'kagglehub>=0.3,<2', 'grad-cam==1.5.7', 'lime==0.2.0.1',
        'scikit-learn>=1.5,<2', 'scipy>=1.13,<2', 'opencv-python-headless>=4.8,<5'])


In [ ]:
import os, sys, json, random, time, warnings, zipfile, platform, hashlib
from pathlib import Path
warnings.filterwarnings("default")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models, transforms as T

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
from scipy import stats

# --- Hiperparámetros (todo lo configurable está aquí) ---
QUICK = os.environ.get("QUICK_RUN", "0") == "1"       # modo de prueba rápida (solo para depurar)
CFG = dict(
    IMG_SIZE=128 if QUICK else 224, BATCH=4, ACCUM_STEPS=1 if QUICK else 8,
    EPOCHS=3 if QUICK else 20, PATIENCE=5,             # early stopping sobre el AUC de validación
    LR=1e-4, WD=1e-4, DROPOUT=0.3,
    SEEDS=[0] if QUICK else [0, 1, 2],                 # 3 semillas -> media ± desviación
    MODELS=["densenet121", "resnet50"],
    N_XAI=8 if QUICK else 40,                          # imágenes de Montgomery para LIME + fidelidad
    LIME_SAMPLES=60 if QUICK else 800,
    PRETRAINED=os.environ.get("NO_PRETRAINED", "0") != "1",
    SPLIT_SEED=42,
)
OUT = Path(os.environ.get("RESULTS_DIR", "results")); OUT.mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 0 if os.name == "nt" else (2 if device.type == "cuda" else 0)
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

print(f"Dispositivo: {device} | torch {torch.__version__} | torchvision {torchvision.__version__}")
if device.type == "cpu" and not QUICK:
    print("AVISO: no hay GPU; el entrenamiento será lento. Activa GPU en Entorno de ejecución.")
print(json.dumps(CFG, indent=1))

## 1. Datos y procedencia
Las etiquetas `_0` y `_1` identifican radiografías normales y radiografías con manifestaciones compatibles con TB según la NLM. El identificador disponible es el de radiografía; no se ha verificado un identificador independiente de paciente. La descarga de imágenes se realiza desde Kaggle; las cajas auxiliares proceden de NLM.


In [ ]:
def find_data_root():
    cands = [os.environ.get("CXR_DATA_ROOT"),
             "/kaggle/input/pulmonary-chest-xray-abnormalities",
             "./pulmonary-chest-xray-abnormalities", "./data"]
    for c in cands:
        if c and Path(c).exists() and any(Path(c).rglob("CXR_png/*.png")):
            return Path(c)
    import kagglehub                                   # descarga pública, sin credenciales
    return Path(kagglehub.dataset_download("kmader/pulmonary-chest-xray-abnormalities"))

ROOT = find_data_root()
print("Dataset en:", ROOT)

rows, masks_l, masks_r = [], {}, {}
for p in ROOT.rglob("*.png"):
    parts = set(p.parts)
    if "leftMask" in parts:  masks_l[p.name] = str(p); continue
    if "rightMask" in parts: masks_r[p.name] = str(p); continue
    if "CXR_png" not in parts: continue
    src = "montgomery" if p.name.startswith("MCUCXR") else ("shenzhen" if p.name.startswith("CHNCXR") else None)
    if src is None: continue
    rows.append(dict(path=str(p), name=p.name, source=src, label=int(p.stem.split("_")[-1])))

df = pd.DataFrame(rows)
assert not df.empty, "No se encontraron radiografías."
assert not df.name.duplicated().any(), "Hay nombres de radiografía duplicados."
df = df.sort_values("name").reset_index(drop=True)
df["has_mask"] = df.name.isin(masks_l) & df.name.isin(masks_r)
assert set(df.source) == {"shenzhen", "montgomery"}, "No se encontraron ambos conjuntos; revisa la ruta del dataset."
assert (df.source.value_counts().to_dict() == {"shenzhen": 662, "montgomery": 138}), "Conteo de imágenes distinto de la fuente NLM."
assert df.loc[df.source == "montgomery", "has_mask"].all(), "Faltan máscaras de Montgomery."
assert set(df.label) == {0, 1}, "Etiquetas inesperadas."

summary = df.groupby(["source", "label"]).size().unstack(fill_value=0).rename(columns={0: "Normal", 1: "TB"})
summary["Total"] = summary.sum(axis=1); summary["con máscara"] = df.groupby("source").has_mask.sum()
display(summary)

In [ ]:
# Caché en memoria (las radiografías originales son enormes: se redimensionan UNA vez)
S = CFG["IMG_SIZE"]
IMG_CACHE = {p: Image.open(p).convert("L").resize((S, S), Image.BILINEAR) for p in tqdm(df.path, desc="Imágenes")}
MASK_CACHE = {}                                        # máscara de pulmón (izq. U der.), solo Montgomery
for n in tqdm(df[df.has_mask].name, desc="Máscaras"):
    l = np.array(Image.open(masks_l[n]).convert("L").resize((S, S), Image.NEAREST)) > 0
    r = np.array(Image.open(masks_r[n]).convert("L").resize((S, S), Image.NEAREST)) > 0
    MASK_CACHE[n] = l | r

# Muestras por fuente y clase (con contorno de pulmón en Montgomery)
fig, axs = plt.subplots(2, 4, figsize=(11, 5.6))
for r, (src, lab) in enumerate([("shenzhen", 0), ("shenzhen", 1)]):
    for c in range(2):
        sub = df[(df.source == src) & (df.label == lab)].iloc[c * 7]
        axs[r, c].imshow(IMG_CACHE[sub.path], cmap="gray"); axs[r, c].set_title(f"Shenzhen · {'TB' if lab else 'Normal'}", fontsize=9)
for r, lab in enumerate([0, 1]):
    for c in range(2):
        sub = df[(df.source == "montgomery") & (df.label == lab)].iloc[c * 5]
        axs[r, 2 + c].imshow(IMG_CACHE[sub.path], cmap="gray")
        if sub.name in MASK_CACHE: axs[r, 2 + c].contour(MASK_CACHE[sub.name], levels=[0.5], colors="yellow", linewidths=0.8)
        axs[r, 2 + c].set_title(f"Montgomery · {'TB' if lab else 'Normal'}", fontsize=9)
for a in axs.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(OUT / "fig_samples.png", dpi=200); plt.show()

## 2. Preprocesamiento y particiones
Se conserva la partición estratificada original de Shenzhen (462/67/133; semilla 42), el redimensionado a 224×224 y la normalización ImageNet. Montgomery permanece fuera del entrenamiento, la parada temprana y la selección de umbral. En modo rápido se reducen resolución y cómputo para verificar el flujo; sus cifras no se mezclan con el estudio.


In [ ]:
sz = df[df.source == "shenzhen"].reset_index(drop=True)
mg = df[df.source == "montgomery"].reset_index(drop=True)

sz_trval, sz_test = train_test_split(sz, test_size=0.20, stratify=sz.label, random_state=CFG["SPLIT_SEED"])
sz_train, sz_val  = train_test_split(sz_trval, test_size=0.125, stratify=sz_trval.label, random_state=CFG["SPLIT_SEED"])
sz_train, sz_val, sz_test = [d.reset_index(drop=True) for d in (sz_train, sz_val, sz_test)]

split_tbl = pd.DataFrame({k: {"n": len(d), "TB (%)": round(100 * d.label.mean(), 1)}
                          for k, d in {"Shenzhen train": sz_train, "Shenzhen val": sz_val,
                                       "Shenzhen test": sz_test, "Montgomery (externo)": mg}.items()}).T
display(split_tbl)
SPLITS = pd.concat([d.assign(split=name) for name, d in [("train", sz_train), ("val", sz_val), ("test", sz_test), ("external", mg)]], ignore_index=True)
assert SPLITS.name.is_unique, "Hay fuga de imágenes entre particiones."
SPLITS[["name", "source", "label", "split"]].to_csv(OUT / "splits.csv", index=False)

train_tf = T.Compose([T.RandomAffine(degrees=8, translate=(0.04, 0.04), scale=(0.95, 1.05)),
                      T.ColorJitter(brightness=0.15, contrast=0.15),
                      T.Grayscale(3), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
eval_tf  = T.Compose([T.Grayscale(3), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

class CXRDataset(Dataset):
    def __init__(self, frame, train=False):
        self.paths, self.y = frame.path.tolist(), frame.label.values
        self.tf = train_tf if train else eval_tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return self.tf(IMG_CACHE[self.paths[i]]), int(self.y[i])

def make_loader(frame, train=False, seed=0, bs=None):
    g = torch.Generator().manual_seed(seed)
    return DataLoader(CXRDataset(frame, train), batch_size=bs or CFG["BATCH"], shuffle=train, generator=g,
                      num_workers=NUM_WORKERS, pin_memory=device.type == "cuda", drop_last=False)

def get_tensor(frame):
    return torch.stack([eval_tf(IMG_CACHE[p]) for p in frame.path])

## 3. Modelos
Dos arquitecturas con *transfer learning* (ImageNet) y una cabeza nueva de 2 clases con *dropout* (regularización):
* **DenseNet121** (Huang et al., 2017): conexiones densas, ~7 M de parámetros.
* **ResNet50** (He et al., 2016): bloques residuales, ~24 M de parámetros.

Se hace *fine-tuning* de toda la red con AdamW (*weight decay* desacoplado), *cosine annealing* y pérdida de entropía cruzada **ponderada por clase**.

In [ ]:
def build_model(name, pretrained=True):
    if name == "densenet121":
        w = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        m = models.densenet121(weights=w)
        m.classifier = nn.Sequential(nn.Dropout(CFG["DROPOUT"]), nn.Linear(m.classifier.in_features, 2))
    elif name == "resnet50":
        w = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        m = models.resnet50(weights=w)
        m.fc = nn.Sequential(nn.Dropout(CFG["DROPOUT"]), nn.Linear(m.fc.in_features, 2))
    else:
        raise ValueError(name)
    return m

def target_layer(model, name):                          # última capa convolucional para Grad-CAM
    return model.features[-1] if name == "densenet121" else model.layer4[-1]

arch_rows = []
for n in CFG["MODELS"]:
    m = build_model(n, pretrained=False)
    n_par = sum(p.numel() for p in m.parameters())
    head = m.classifier if n == "densenet121" else m.fc
    arch_rows.append(dict(modelo=n, parametros_M=round(n_par / 1e6, 2),
                          capa_Grad_CAM=("features[-1] (norm5)" if n == "densenet121" else "layer4[-1]"),
                          cabeza=str(head).replace("\n", " ")))
display(pd.DataFrame(arch_rows))
PARAMS_M = {r["modelo"]: r["parametros_M"] for r in arch_rows}

## 4. Entrenamiento
Registro de pérdida y exactitud por época, selección del mejor modelo por **AUC de validación** y *early stopping*. El umbral de decisión se elige en validación (índice de Youden) y **se mantiene fijo** al evaluar en el test y en Montgomery.

In [ ]:
def run_epoch(model, loader, criterion, opt=None, scaler=None):
    train = opt is not None
    model.train(train)
    total, n, correct, probs, ys = 0.0, 0, 0, [], []
    if train:
        opt.zero_grad(set_to_none=True)
    with torch.set_grad_enabled(train):
        for step, (x, y) in enumerate(loader):
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16,
                                enabled=train and device.type == "cuda"):
                out = model(x)
                loss = criterion(out, y)
            if train:
                group_start = (step // CFG["ACCUM_STEPS"]) * CFG["ACCUM_STEPS"]
                group_size = min(CFG["ACCUM_STEPS"], len(loader) - group_start)
                scaler.scale(loss / group_size).backward()
                if (step + 1) % CFG["ACCUM_STEPS"] == 0 or step + 1 == len(loader):
                    scaler.step(opt)
                    scaler.update()
                    opt.zero_grad(set_to_none=True)
            total += loss.item() * len(y)
            n += len(y)
            correct += (out.argmax(1) == y).sum().item()
            probs.append(F.softmax(out.float(), 1)[:, 1].detach().cpu().numpy())
            ys.append(y.cpu().numpy())
    probs, ys = np.concatenate(probs), np.concatenate(ys)
    auc = roc_auc_score(ys, probs) if len(np.unique(ys)) > 1 else float("nan")
    return total / n, correct / n, auc

def train_model(name, seed):
    set_seed(seed)
    model = build_model(name, CFG["PRETRAINED"]).to(device)
    cnt = np.bincount(sz_train.label.values, minlength=2)
    weights = torch.tensor(len(sz_train) / (2.0 * cnt), dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WD"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["EPOCHS"])
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    dl_tr, dl_va = make_loader(sz_train, True, seed), make_loader(sz_val, False)
    hist = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc", "val_auc"]}
    best_auc, best_ep, bad = -1, 0, 0
    ckpt = OUT / f"{name}_s{seed}.pt"
    for ep in range(CFG["EPOCHS"]):
        tl, ta, _ = run_epoch(model, dl_tr, criterion, opt, scaler)
        sched.step()
        vl, va, vauc = run_epoch(model, dl_va, criterion)
        for k, v in zip(hist, [tl, ta, vl, va, vauc]):
            hist[k].append(float(v))
        if vauc > best_auc:
            best_auc, best_ep, bad = vauc, ep, 0
            torch.save(model.state_dict(), ckpt)
        else:
            bad += 1
        print(f"  [{name} s{seed}] ep {ep+1:02d} | loss {tl:.3f}/{vl:.3f} | acc {ta:.3f}/{va:.3f} | val AUC {vauc:.3f}", flush=True)
        if bad >= CFG["PATIENCE"]:
            print("  early stopping")
            break
    model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
    return model, hist, best_ep + 1

@torch.no_grad()
def predict_proba(model, frame):
    model.eval()
    ps = []
    for x, _ in make_loader(frame, False, bs=8):
        ps.append(F.softmax(model(x.to(device)), 1)[:, 1].cpu().numpy())
    return np.concatenate(ps), frame.label.values

def youden_threshold(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    return float(thr[np.argmax(tpr - fpr)])

def compute_metrics(y, p, thr):
    yhat = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    sens, spec = tp / max(tp + fn, 1), tn / max(tn + fp, 1)
    prec = tp / max(tp + fp, 1)
    return dict(acc=(tp + tn) / len(y), sens=sens, spec=spec, prec=prec,
                f1=2 * prec * sens / max(prec + sens, 1e-12), auc=roc_auc_score(y, p))


In [ ]:
RUNS = {}
# Se reutiliza una corrida completa solo si la configuración y el manifiesto
# de particiones coinciden y están todos los archivos requeridos.
checkpoint_signature = dict(cfg=CFG, splits_sha256=hashlib.sha256((OUT / 'splits.csv').read_bytes()).hexdigest())
signature_file = OUT / 'checkpoint_signature.json'
history_file = OUT / 'training_history.json'
saved_signature = json.loads(signature_file.read_text(encoding='utf-8')) if signature_file.exists() else None
can_resume = (saved_signature == checkpoint_signature and history_file.exists() and
              all((OUT / f'{m}_s{s}.pt').exists() for m in CFG['MODELS'] for s in CFG['SEEDS']))
saved_history = json.loads(history_file.read_text(encoding='utf-8')) if can_resume else {}
signature_file.write_text(json.dumps(checkpoint_signature, indent=2), encoding='utf-8')
print('Reanudación verificada:', can_resume)
t_all = time.time()
for mname in CFG["MODELS"]:
    for seed in CFG["SEEDS"]:
        t0 = time.time(); print(f"\n=== {mname} | semilla {seed} ===")
        if can_resume:
            model = build_model(mname, pretrained=False).to(device)
            model.load_state_dict(torch.load(OUT / f'{mname}_s{seed}.pt', map_location=device, weights_only=True))
            model.eval()
            hist = saved_history[f'{mname}_s{seed}']
            best_ep = int(np.argmax(hist['val_auc'])) + 1
        else:
            model, hist, best_ep = train_model(mname, seed)
        pv, yv = predict_proba(model, sz_val); thr = youden_threshold(yv, pv)
        pt, yt = predict_proba(model, sz_test); pm, ym = predict_proba(model, mg)
        RUNS[(mname, seed)] = dict(hist=hist, thr=thr, best_epoch=best_ep,
                                   p_sz=pt, y_sz=yt, p_mg=pm, y_mg=ym,
                                   m_sz=compute_metrics(yt, pt, thr), m_mg=compute_metrics(ym, pm, thr))
        r = RUNS[(mname, seed)]
        print(f"  -> thr={thr:.3f} | Shenzhen AUC {r['m_sz']['auc']:.3f} F1 {r['m_sz']['f1']:.3f} | "
              f"Montgomery AUC {r['m_mg']['auc']:.3f} F1 {r['m_mg']['f1']:.3f} | {time.time()-t0:.0f}s")
        del model; torch.cuda.empty_cache()
print(f"\nTiempo total de entrenamiento: {(time.time()-t_all)/60:.1f} min")

## 5. Evaluación
Métricas en el test interno (Shenzhen) y en la validación externa (Montgomery), media ± desviación estándar sobre las semillas. **Sens.** = sensibilidad (recall de TB), **Espec.** = especificidad.

In [ ]:
METRICS = [("acc", "Acc"), ("f1", "F1"), ("sens", "Sens."), ("spec", "Espec."), ("auc", "AUC")]
MODEL_LABEL = {"densenet121": "DenseNet121", "resnet50": "ResNet50"}
SET_LABEL = {"sz": "Shenzhen (test)", "mg": "Montgomery (ext.)"}

def perf_stats(mname, sset):
    vals = {k: np.array([RUNS[(mname, s)][f"m_{sset}"][k] for s in CFG["SEEDS"]]) for k, _ in METRICS}
    return {k: (v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0) for k, v in vals.items()}

perf_rows = []
for mname in CFG["MODELS"]:
    for sset in ("sz", "mg"):
        st = perf_stats(mname, sset)
        perf_rows.append({"Modelo": MODEL_LABEL[mname], "Conjunto": SET_LABEL[sset],
                          **{lab: f"{st[k][0]:.3f} ± {st[k][1]:.3f}" for k, lab in METRICS}})
perf_df = pd.DataFrame(perf_rows); display(perf_df)

In [ ]:
SEED0 = CFG["SEEDS"][0]
colors = {"densenet121": "tab:blue", "resnet50": "tab:red"}

# (a) Curvas de entrenamiento (semilla 0)
fig, axs = plt.subplots(1, 2, figsize=(9, 3.2))
for mname in CFG["MODELS"]:
    h = RUNS[(mname, SEED0)]["hist"]; ep = np.arange(1, len(h["train_loss"]) + 1)
    axs[0].plot(ep, h["train_loss"], "--", c=colors[mname], label=f"{MODEL_LABEL[mname]} train")
    axs[0].plot(ep, h["val_loss"], "-", c=colors[mname], label=f"{MODEL_LABEL[mname]} val")
    axs[1].plot(ep, h["val_auc"], "-o", ms=3, c=colors[mname], label=MODEL_LABEL[mname])
axs[0].set(xlabel="Época", ylabel="Pérdida (CE ponderada)", title="Pérdida"); axs[0].legend(fontsize=7)
axs[1].set(xlabel="Época", ylabel="AUC de validación", title="AUC en validación"); axs[1].legend(fontsize=7)
for a in axs: a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT / "fig_curves.png", dpi=200); plt.show()

# (b) Curvas ROC (semilla 0): interno vs. externo
fig, axs = plt.subplots(1, 2, figsize=(8, 3.4), sharey=True)
for ax, sset in zip(axs, ("sz", "mg")):
    for mname in CFG["MODELS"]:
        r = RUNS[(mname, SEED0)]; fpr, tpr, _ = roc_curve(r[f"y_{sset}"], r[f"p_{sset}"])
        ax.plot(fpr, tpr, c=colors[mname], label=f"{MODEL_LABEL[mname]} (AUC={r[f'm_{sset}']['auc']:.3f})")
    ax.plot([0, 1], [0, 1], "k:", lw=.8); ax.set(title=SET_LABEL[sset], xlabel="1 − Especificidad"); ax.grid(alpha=.3); ax.legend(fontsize=7, loc="lower right")
axs[0].set_ylabel("Sensibilidad")
plt.tight_layout(); plt.savefig(OUT / "fig_roc.png", dpi=200); plt.show()

# (c) Matrices de confusión (semilla 0)
fig, axs = plt.subplots(2, 2, figsize=(6, 5.2))
for i, mname in enumerate(CFG["MODELS"]):
    for j, sset in enumerate(("sz", "mg")):
        r = RUNS[(mname, SEED0)]
        ConfusionMatrixDisplay(confusion_matrix(r[f"y_{sset}"], (r[f"p_{sset}"] >= r["thr"]).astype(int), labels=[0, 1]),
                               display_labels=["Normal", "TB"]).plot(ax=axs[i, j], colorbar=False, cmap="Blues")
        axs[i, j].set_title(f"{MODEL_LABEL[mname]} · {SET_LABEL[sset]}", fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "fig_confusion.png", dpi=200); plt.show()

## 6. Explicabilidad (XAI): ¿mira el modelo los pulmones?
Se explican los modelos de la semilla 0, siempre para la **clase predicha**:
* **Grad-CAM** (Selvaraju et al., 2017): gradientes de la clase respecto a la última capa convolucional (`pytorch-grad-cam`).
* **LIME** (Ribeiro et al., 2016): sustituto lineal local sobre superpíxeles SLIC; el mapa conserva los pesos positivos (evidencia a favor de la clase predicha).
* **Aleatorio**: mapa de ruido suavizado (hipótesis nula).

**Métricas cuantitativas** (mapa normalizado a [0, 1]):
| Métrica | Definición | Lectura |
|---|---|---|
| **EIL** (*energy in lung*) | fracción de la energía del mapa dentro de la máscara de pulmón | más alto = más centrado en el pulmón |
| **Ganancia** | EIL / fracción de área pulmonar | 1 = como un mapa uniforme; > 1 = prefiere el pulmón |
| **Puntería** | ¿el máximo del mapa cae dentro del pulmón? | proporción de aciertos |
| **Borde** | energía en el marco exterior (5 %) / área del marco | > 1 = el modelo atiende a los márgenes (posibles artefactos); no necesita máscara |
| **Deletion AUC** (fidelidad) | probabilidad media de la clase al ocultar el 0–50 % de píxeles más relevantes según el mapa | más bajo = explicación más fiel (Petsiuk et al., 2018) |

Las métricas con máscara solo se pueden calcular en Montgomery; **Borde** también en el test de Shenzhen.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from lime import lime_image
from lime.wrappers.scikit_image import SegmentationAlgorithm

MEAN_T = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1)
STD_T = torch.tensor(IMAGENET_STD, device=device).view(1, 3, 1, 1)
BW = int(0.05 * S); FRAME = np.ones((S, S), bool); FRAME[BW:-BW, BW:-BW] = False; FRAME_FRAC = FRAME.mean()

def load_trained(mname, seed):
    m = build_model(mname, pretrained=False)
    m.load_state_dict(torch.load(OUT / f"{mname}_s{seed}.pt", map_location=device, weights_only=True))
    return m.to(device).eval()

@torch.no_grad()
def class_probs(model, X):
    return torch.cat([F.softmax(model(X[i:i + 8].to(device)), 1).cpu() for i in range(0, len(X), 8)]).numpy()

def gradcam_maps(model, mname, X, classes, bs=2):
    cam = GradCAM(model=model, target_layers=[target_layer(model, mname)])
    out = []
    for i in range(0, len(X), bs):
        tg = [ClassifierOutputTarget(int(c)) for c in classes[i:i + bs]]
        out.append(cam(input_tensor=X[i:i + bs].to(device), targets=tg))
    try: cam.activations_and_grads.release()
    except Exception: pass
    return np.concatenate(out)                           # (N, S, S) en [0, 1]

def lime_maps(model, frame, classes, n_samples):
    @torch.no_grad()
    def predict_fn(batch):                               # batch: (n, H, W, 3) uint8 -> probabilidades
        x = torch.from_numpy(np.asarray(batch, dtype=np.float32) / 255.0).permute(0, 3, 1, 2).to(device)
        return F.softmax(model((x - MEAN_T) / STD_T), 1).cpu().numpy()
    explainer = lime_image.LimeImageExplainer(random_state=0)
    seg = SegmentationAlgorithm("slic", n_segments=60, compactness=10, sigma=1, start_label=1)
    maps = []
    for path, c in tqdm(list(zip(frame.path, classes)), desc="LIME", leave=False):
        rgb = np.stack([np.array(IMG_CACHE[path])] * 3, -1).astype(np.uint8)
        kw = dict(top_labels=2, hide_color=0, num_samples=n_samples, batch_size=8, segmentation_fn=seg)
        try:
            exp = explainer.explain_instance(rgb, predict_fn, progress_bar=False, **kw)
        except TypeError:                                # versiones antiguas de lime no tienen progress_bar
            exp = explainer.explain_instance(rgb, predict_fn, **kw)
        w = dict(exp.local_exp[int(c)]); h = np.zeros(exp.segments.shape, np.float32)
        for sid in np.unique(exp.segments): h[exp.segments == sid] = max(w.get(sid, 0.0), 0.0)
        maps.append(h / h.max() if h.max() > 0 else h)
    return np.stack(maps)

def random_maps(n, seed=0):
    rng = np.random.RandomState(seed); out = []
    for _ in range(n):
        h = cv2.resize(rng.rand(7, 7).astype(np.float32), (S, S), interpolation=cv2.INTER_LINEAR)
        out.append((h - h.min()) / (h.max() - h.min() + 1e-12))
    return np.stack(out)

def score_map(h, mask=None):
    h = np.clip(h, 0, None); tot = h.sum()
    area = float(mask.mean()) if mask is not None else np.nan
    if tot <= 1e-12:
        return dict(border=np.nan, eil=np.nan, area=area, gain=np.nan, point=np.nan, valid=False)
    r = dict(border=(h[FRAME].sum() / tot) / FRAME_FRAC, eil=np.nan, area=area, gain=np.nan, point=np.nan, valid=True)
    if mask is not None:
        if area <= 0:
            raise ValueError("M?scara pulmonar vac?a")
        eil = h[mask].sum() / tot
        r.update(eil=eil, gain=eil / area, point=float(mask[np.unravel_index(h.argmax(), h.shape)]))
    return r

@torch.no_grad()
def deletion_auc(model, x, h, cls, fracs=np.linspace(0, 0.5, 11), seed=0):
    if np.max(h) <= 0:
        return float("nan")
    rng = np.random.RandomState(seed)
    gray = (x[0].cpu().numpy() * IMAGENET_STD[0] + IMAGENET_MEAN[0]).astype(np.float32)
    blurred = cv2.GaussianBlur(gray, (0, 0), sigmaX=10)
    background = torch.stack([(torch.from_numpy(blurred) - IMAGENET_MEAN[c]) / IMAGENET_STD[c] for c in range(3)])
    order = np.argsort(-(h.ravel() + 1e-6 * rng.rand(h.size)))        # más relevante primero (desempate aleatorio)
    batch = []
    for f in fracs:
        xx = x.clone(); k = int(f * h.size)
        if k > 0:
            rr, cc = np.unravel_index(order[:k], h.shape); xx[:, rr, cc] = background[:, rr, cc]   # fondo gris desenfocado, normalizado por canal
        batch.append(xx)
    p = F.softmax(model(torch.stack(batch).to(device)), 1)[:, cls].cpu().numpy()
    return float(np.sum((p[1:] + p[:-1]) / 2 * np.diff(fracs)) / (fracs[-1] - fracs[0]))

In [ ]:
# Subconjunto de Montgomery (equilibrado por clase) para LIME + fidelidad
rng = np.random.RandomState(0); ids = []
for c in (0, 1):
    pool = mg.index[mg.label == c].values
    ids += rng.choice(pool, size=min(CFG["N_XAI"] // 2, len(pool)), replace=False).tolist()
mg_sub = mg.loc[ids].reset_index(drop=True)
print("Subconjunto XAI:", len(mg_sub), "imágenes |", mg_sub.label.value_counts().to_dict())

X_mg, X_sub, X_szt = get_tensor(mg), get_tensor(mg_sub), get_tensor(sz_test)
XAI = {}           # resultados por modelo
sub_rows, full_rows = [], []

for mname in CFG["MODELS"]:
    t0 = time.time(); model = load_trained(mname, SEED0); thr = RUNS[(mname, SEED0)]["thr"]
    pred = lambda X: (class_probs(model, X)[:, 1] >= thr).astype(int)       # clase predicha con el umbral de validación
    c_sub, c_mg, c_szt = pred(X_sub), pred(X_mg), pred(X_szt)

    # --- Subconjunto: Grad-CAM, LIME y aleatorio + fidelidad (deletion) ---
    maps = {"Grad-CAM": gradcam_maps(model, mname, X_sub, c_sub),
            "LIME": lime_maps(model, mg_sub, c_sub, CFG["LIME_SAMPLES"]),
            "Aleatorio": random_maps(len(mg_sub))}
    for meth, M in maps.items():
        for i in range(len(mg_sub)):
            sc = score_map(M[i], MASK_CACHE.get(mg_sub.name[i]))
            sc.update(model=mname, method=meth, name=mg_sub.name[i], label=int(mg_sub.label[i]), pred=int(c_sub[i]),
                      deletion=deletion_auc(model, X_sub[i], M[i], int(c_sub[i])))
            sub_rows.append(sc)

    # --- Grad-CAM sobre TODO Montgomery y el test de Shenzhen ---
    G_mg = gradcam_maps(model, mname, X_mg, c_mg); G_sz = gradcam_maps(model, mname, X_szt, c_szt)
    for i in range(len(mg)):
        sc = score_map(G_mg[i], MASK_CACHE.get(mg.name[i])); sc.update(model=mname, set="mg", name=mg.name[i], label=int(mg.label[i]), pred=int(c_mg[i])); full_rows.append(sc)
    for i in range(len(sz_test)):
        sc = score_map(G_sz[i], None); sc.update(model=mname, set="sz", name=sz_test.name[i], label=int(sz_test.label[i]), pred=int(c_szt[i])); full_rows.append(sc)

    XAI[mname] = dict(maps=maps, G_mg=G_mg, c_sub=c_sub)
    print(f"{mname}: XAI listo en {time.time()-t0:.0f}s"); del model; torch.cuda.empty_cache()

sub_df, full_df = pd.DataFrame(sub_rows), pd.DataFrame(full_rows)
full_df["correct"] = full_df.pred == full_df.label; sub_df["correct"] = sub_df.pred == sub_df.label

In [ ]:
def mean_sd(s):
    s = pd.Series(s).dropna(); return (s.mean(), s.std(ddof=1) if len(s) > 1 else 0.0) if len(s) else (np.nan, np.nan)
fmt = lambda t, d=2: "--" if np.isnan(t[0]) else f"{t[0]:.{d}f} ± {t[1]:.{d}f}"

XAI_TABLE = []                                                       # filas para pantalla y LaTeX
for mname in CFG["MODELS"]:
    for meth in ["Grad-CAM", "LIME", "Aleatorio"]:
        d = sub_df[(sub_df.model == mname) & (sub_df.method == meth)]
        XAI_TABLE.append(dict(Modelo=MODEL_LABEL[mname], Método=meth, n=int(d.border.notna().sum()), EIL=mean_sd(d.eil), Ganancia=mean_sd(d.gain),
                              Puntería=(d.point.mean(), 0.0), Borde=mean_sd(d.border), Deletion=mean_sd(d.deletion)))
    for sset, lab in (("mg", "Grad-CAM (Mont. completo)"), ("sz", "Grad-CAM (Shen. test)")):
        d = full_df[(full_df.model == mname) & (full_df.set == sset)]
        XAI_TABLE.append(dict(Modelo=MODEL_LABEL[mname], Método=lab, n=int(d.border.notna().sum()), EIL=mean_sd(d.eil), Ganancia=mean_sd(d.gain),
                              Puntería=(d.point.mean() if sset == "mg" else np.nan, 0.0), Borde=mean_sd(d.border), Deletion=(np.nan, np.nan)))

disp = pd.DataFrame([{**{k: v for k, v in r.items() if k in ("Modelo", "Método", "n")},
                      "EIL": fmt(r["EIL"], 3), "Ganancia": fmt(r["Ganancia"]),
                      "Puntería": "--" if np.isnan(r["Puntería"][0]) else f"{r['Puntería'][0]:.2f}",
                      "Borde": fmt(r["Borde"]), "Deletion↓": fmt(r["Deletion"], 3)} for r in XAI_TABLE])
display(disp)
area_mean = float(np.mean([m.mean() for m in MASK_CACHE.values()]))
print(f"Fracción media de área pulmonar en Montgomery (nivel del azar para EIL): {area_mean:.3f}")

# Pruebas estadísticas
STATS = {}
for mname in CFG["MODELS"]:
    g = sub_df[(sub_df.model == mname) & (sub_df.method == "Grad-CAM")].set_index("name").eil
    l = sub_df[(sub_df.model == mname) & (sub_df.method == "LIME")].set_index("name").eil.reindex(g.index)
    pairs = pd.concat([g.rename("gradcam"), l.rename("lime")], axis=1).dropna()
    p_w = stats.wilcoxon(pairs.gradcam, pairs.lime).pvalue if len(pairs) > 5 and (pairs.gradcam - pairs.lime).abs().sum() > 0 else float("nan")
    f = full_df[(full_df.model == mname) & (full_df.set == "mg")]
    ok, err = f[f.correct].eil.dropna(), f[~f.correct].eil.dropna()
    p_mw = stats.mannwhitneyu(ok, err).pvalue if len(err) >= 3 else float("nan")
    STATS[mname] = dict(p_wilcoxon=p_w, p_mw=p_mw, eil_ok=ok.mean(), eil_err=err.mean() if len(err) else np.nan, n_ok=len(ok), n_err=len(err))
    print(f"{MODEL_LABEL[mname]}: Wilcoxon Grad-CAM vs LIME (EIL) p={p_w:.4f} | aciertos n={len(ok)} EIL={ok.mean():.3f} | "
          f"errores n={len(err)} EIL={(err.mean() if len(err) else float('nan')):.3f} | Mann-Whitney p={p_mw:.4f}")

# EIL por clase real (Grad-CAM, Montgomery completo)
display(full_df[full_df.set == "mg"].groupby(["model", "label"]).agg(n=("eil", "size"), EIL=("eil", "mean"),
        Ganancia=("gain", "mean"), Borde=("border", "mean")).rename(index={0: "Normal", 1: "TB"}, level=1))

In [ ]:
# --- Figura cualitativa: Grad-CAM y LIME (modelo 1) ---
def overlay(ax, img, h, mask=None, title=""):
    ax.imshow(img, cmap="gray"); ax.imshow(h, cmap="jet", alpha=np.clip(h, 0, 1) * 0.55, vmin=0, vmax=1)
    if mask is not None: ax.contour(mask, levels=[0.5], colors="white", linewidths=0.7)
    ax.set_title(title, fontsize=7); ax.axis("off")

mname = CFG["MODELS"][0]; maps = XAI[mname]["maps"]; c_sub = XAI[mname]["c_sub"]
tb_ok = np.where((mg_sub.label == 1) & (c_sub == 1))[0]; nm_ok = np.where((mg_sub.label == 0) & (c_sub == 0))[0]
wrong = np.where(mg_sub.label.values != c_sub)[0]
picks = [("TB acertado", tb_ok[0] if len(tb_ok) else None), ("Normal acertado", nm_ok[0] if len(nm_ok) else None),
         ("Error", wrong[0] if len(wrong) else (tb_ok[1] if len(tb_ok) > 1 else None))]
picks = [(t, i) for t, i in picks if i is not None]
fig, axs = plt.subplots(len(picks), 3, figsize=(5.2, 1.8 * len(picks)))
axs = np.atleast_2d(axs)
for r, (tag, i) in enumerate(picks):
    nm = mg_sub.name[i]; mk = MASK_CACHE.get(nm); img = IMG_CACHE[mg_sub.path[i]]
    lab = f"real={'TB' if mg_sub.label[i] else 'N'}, pred={'TB' if c_sub[i] else 'N'}"
    axs[r, 0].imshow(img, cmap="gray"); axs[r, 0].contour(mk, levels=[0.5], colors="yellow", linewidths=0.7) if mk is not None else None
    axs[r, 0].set_title(f"{tag}\n{lab}", fontsize=7); axs[r, 0].axis("off")
    overlay(axs[r, 1], img, maps["Grad-CAM"][i], mk, "Grad-CAM"); overlay(axs[r, 2], img, maps["LIME"][i], mk, "LIME")
plt.tight_layout(); plt.savefig(OUT / "fig_cam_examples.png", dpi=220); plt.show()

# --- Figura cuantitativa: EIL por método (el punteado es el nivel del azar) ---
fig, axs = plt.subplots(1, 2, figsize=(5.4, 2.6), sharey=True)
for ax, mname in zip(axs, CFG["MODELS"]):
    data = [sub_df[(sub_df.model == mname) & (sub_df.method == m)].eil.dropna().values for m in ["Grad-CAM", "LIME", "Aleatorio"]]
    ax.boxplot(data, showmeans=True); ax.set_xticklabels(["Grad-CAM", "LIME", "Aleat."])
    ax.axhline(area_mean, ls="--", c="gray", lw=1); ax.set_title(MODEL_LABEL[mname], fontsize=8); ax.tick_params(labelsize=7); ax.grid(alpha=.3)
axs[0].set_ylabel("EIL (energía en pulmón)", fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "fig_xai_box.png", dpi=220); plt.show()

In [ ]:
# Anotaciones oficiales de NLM: descarga directa o CSV ya presentes/cargados.
import hashlib, urllib.request
DATA_DIR = Path('data'); DATA_DIR.mkdir(exist_ok=True)
ROI_URLS = {
 'montgomery_consensus_roi.csv': 'https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Montgomery-County-CXR-Set/MontgomerySet/montgomery_consensus_roi.csv',
 'shenzhen_consensus_roi.csv': 'https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Shenzhen-Hospital-CXR-Set/shenzhen_consensus_roi.csv'}
ROI_STATUS = {}
for name, url in ROI_URLS.items():
    path = DATA_DIR / name
    if not path.exists():
        try: urllib.request.urlretrieve(url, path)
        except Exception as exc: ROI_STATUS[name] = 'PENDIENTE: ' + str(exc)
    if path.exists():
        table = pd.read_csv(path)
        assert {'patientId','x_dis','y_dis','width_dis','height_dis','Labels'} <= set(table)
        ROI_STATUS[name] = dict(url=url, boxes=len(table), images=table.patientId.nunique(),
                                sha256=hashlib.sha256(path.read_bytes()).hexdigest())
print(json.dumps(ROI_STATUS, indent=2, ensure_ascii=False))
if not (DATA_DIR / 'montgomery_consensus_roi.csv').exists():
    raise FileNotFoundError('Cargue el CSV oficial NLM en data/ y vuelva a ejecutar desde esta celda.')


## 7. Localización histórica de cajas de hallazgos
Se explica la clase positiva fija, incluso en falsos negativos. Este análisis de cajas es distinto del análisis general anterior, que explicaba la clase predicha. Las cajas son aproximaciones anatómicas, no segmentaciones de lesión.


In [ ]:


import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats


def boxes_to_mask(rows, size, original_size=1024):
    """Union rectangular consensus regions in the notebook's square image grid."""
    mask = np.zeros((size, size), dtype=bool)
    for row in rows.itertuples(index=False):
        x0 = max(0, min(size, int(np.floor(row.x_dis * size / original_size))))
        y0 = max(0, min(size, int(np.floor(row.y_dis * size / original_size))))
        x1 = max(0, min(size, int(np.ceil((row.x_dis + row.width_dis) * size / original_size))))
        y1 = max(0, min(size, int(np.ceil((row.y_dis + row.height_dis) * size / original_size))))
        mask[y0:y1, x0:x1] = True
    return mask


def roi_scores(heatmap, lesion_mask, lung_mask):
    h = np.clip(np.asarray(heatmap, dtype=np.float64), 0, None)
    lesion = np.asarray(lesion_mask, dtype=bool)
    lung = np.asarray(lung_mask, dtype=bool)
    if h.shape != lesion.shape or h.shape != lung.shape:
        raise ValueError("Los mapas y las máscaras tienen dimensiones distintas")
    if h.sum() <= 0 or not lesion.any() or not lung.any():
        return dict(gain_image=np.nan, gain_lung=np.nan, point_lung=np.nan,
                    lesion_area_image=float(lesion.mean()), lesion_area_lung=np.nan,
                    roi_inside_lung_fraction=np.nan)
    gain_image = (h[lesion].sum() / h.sum()) / lesion.mean()
    lesion_in_lung = lesion & lung
    area_lung = lesion_in_lung.sum() / lung.sum()
    if h[lung].sum() <= 0 or area_lung <= 0:
        gain_lung = point_lung = np.nan
    else:
        gain_lung = (h[lesion_in_lung].sum() / h[lung].sum()) / area_lung
        candidate = np.where(lung, h, -np.inf)
        point_lung = float(lesion_in_lung.flat[int(candidate.argmax())])
    return dict(gain_image=float(gain_image), gain_lung=float(gain_lung),
                point_lung=float(point_lung), lesion_area_image=float(lesion.mean()),
                lesion_area_lung=float(area_lung),
                roi_inside_lung_fraction=float(lesion_in_lung.sum() / lesion.sum()))


def bootstrap_mean(values, seed=371, n_resamples=3000):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if not len(values):
        return [None, None, None]
    rng = np.random.default_rng(seed)
    means = values[rng.integers(0, len(values), size=(n_resamples, len(values)))].mean(axis=1)
    return [float(values.mean()), float(np.quantile(means, 0.025)), float(np.quantile(means, 0.975))]


def run_lesion_analysis(*, csv_path, mg, mask_cache, get_tensor,
                        load_trained, gradcam_maps, random_maps, class_probs,
                        model_names, seeds, size, out_dir):
    csv_path, out_dir = Path(csv_path), Path(out_dir)
    annotations = pd.read_csv(csv_path)
    expected = {"patientId", "x_dis", "y_dis", "width_dis", "height_dis", "Labels"}
    if not expected.issubset(annotations.columns):
        raise ValueError("Columnas inesperadas en las anotaciones NLM")
    if set(annotations.Labels) != {"TB"}:
        raise ValueError("Las anotaciones contienen etiquetas inesperadas")
    frame = mg[mg.name.isin(annotations.patientId)].reset_index(drop=True)
    if len(frame) != 55 or not (frame.label == 1).all():
        raise ValueError(f"Se esperaban 55 radiografías positivas anotadas, hay {len(frame)}")
    if not all(name in mask_cache for name in frame.name):
        raise ValueError("Faltan máscaras pulmonares para el análisis de lesiones")
    masks = {name: boxes_to_mask(annotations[annotations.patientId == name], size)
             for name in frame.name}
    if not all(mask.any() for mask in masks.values()):
        raise ValueError("Hay una máscara de lesión vacía")
    X = get_tensor(frame)
    targets = np.ones(len(frame), dtype=int)  # Misma clase TB aun en falsos negativos.
    rows = []
    for model_name in model_names:
        for seed in seeds:
            model = load_trained(model_name, seed)
            probabilities = class_probs(model, X)[:, 1]
            maps = gradcam_maps(model, model_name, X, targets)
            controls = random_maps(len(frame), seed=seed + 919)
            for i, item in frame.iterrows():
                for method, heatmap in (("Grad-CAM", maps[i]), ("Aleatorio", controls[i])):
                    metrics = roi_scores(heatmap, masks[item["name"]], mask_cache[item["name"]])
                    rows.append(dict(model=model_name, seed=int(seed), method=method,
                                     name=item["name"], probability_tb=float(probabilities[i]),
                                     **metrics))
            del model
    scores = pd.DataFrame(rows)
    scores.to_csv(out_dir / "lesion_roi_scores.csv", index=False)
    per_image = scores.groupby(["model", "method", "name"], as_index=False).agg(
        gain_image=("gain_image", "mean"), gain_lung=("gain_lung", "mean"),
        point_lung=("point_lung", "mean"), lesion_area_lung=("lesion_area_lung", "mean"),
        roi_inside_lung_fraction=("roi_inside_lung_fraction", "mean"))
    n_misaligned = sum((masks[name] & mask_cache[name]).sum() / masks[name].sum() < .5
                       for name in frame.name)
    result = {"n_images": len(frame), "n_boxes": len(annotations),
              "n_roi_majority_outside_lung": int(n_misaligned), "models": {}}
    for model_name in model_names:
        model_result = {}
        for method in ("Grad-CAM", "Aleatorio"):
            part = per_image[(per_image.model == model_name) & (per_image.method == method)]
            raw_part = scores[(scores.model == model_name) & (scores.method == method)]
            zero_included = raw_part.assign(gain_lung=raw_part.gain_lung.fillna(0)).groupby("name").gain_lung.mean()
            model_result[method] = {
                "gain_lung": bootstrap_mean(part.gain_lung),
                "gain_lung_null_maps_as_zero": bootstrap_mean(zero_included),
                "gain_image": bootstrap_mean(part.gain_image),
                "point_lung": bootstrap_mean(part.point_lung),
                "mean_roi_lung_area": float(part.lesion_area_lung.mean()),
                "valid_images": int(part.gain_lung.notna().sum()),
                "valid_maps": int(raw_part.gain_lung.notna().sum()),
                "total_maps": int(len(raw_part)),
            }
        wide = per_image[per_image.model == model_name].pivot(index="name", columns="method", values="gain_lung")
        differences = (wide["Grad-CAM"] - wide["Aleatorio"]).dropna()
        model_result["paired_gain_lung_difference"] = bootstrap_mean(differences)
        model_result["wilcoxon_p"] = float(stats.wilcoxon(differences).pvalue) if len(differences) > 5 else None
        valid_names = per_image[(per_image.model == model_name) & (per_image.method == "Grad-CAM") &
                                (per_image.roi_inside_lung_fraction >= .5)].name
        model_result["gain_lung_excluding_misaligned_roi"] = bootstrap_mean(
            per_image[(per_image.model == model_name) & (per_image.method == "Grad-CAM") &
                      (per_image.name.isin(valid_names))].gain_lung)
        result["models"][model_name] = model_result
    (out_dir / "lesion_roi_summary.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    fig, axs = plt.subplots(1, len(model_names), figsize=(7.4, 3.5), sharey=True)
    axs = np.atleast_1d(axs)
    for ax, model_name in zip(axs, model_names):
        values = [per_image[(per_image.model == model_name) & (per_image.method == method)].gain_lung.dropna()
                  for method in ("Grad-CAM", "Aleatorio")]
        ax.boxplot(values, tick_labels=["Grad-CAM", "Aleatorio"], showmeans=True)
        ax.axhline(1, color="gray", linestyle="--", linewidth=1)
        ax.set_title("DenseNet121" if model_name == "densenet121" else "ResNet50")
        ax.grid(alpha=.25)
    axs[0].set_ylabel("Ganancia en lesión / azar")
    fig.tight_layout(pad=0.8)
    fig.savefig(out_dir / "fig_lesion_roi.png", dpi=220)
    plt.close(fig)
    return result


In [ ]:
LESION_ROI = run_lesion_analysis(csv_path="data/montgomery_consensus_roi.csv", mg=mg, mask_cache=MASK_CACHE,
    get_tensor=get_tensor, load_trained=load_trained, gradcam_maps=gradcam_maps,
    random_maps=random_maps, class_probs=class_probs, model_names=CFG["MODELS"],
    seeds=CFG["SEEDS"], size=S, out_dir=OUT)
display(pd.DataFrame(LESION_ROI["models"]).T)


## 7. Exportación para el artículo
Se generan las tablas LaTeX, un archivo de macros con **todas las cifras** del artículo y un `.zip` con los resultados. Copia la carpeta `results/` junto a `articulo_ieee.tex` y compila: los números del artículo se actualizan solos.

In [ ]:
MK = {"densenet121": "Dense", "resnet50": "Res"}; SK = {"sz": "Sz", "mg": "Mg"}
MACROS = {}
pm = lambda m, s, d=2: f"{m:.{d}f}$\\pm${s:.{d}f}"

# Datos
MACROS.update(nSzTrain=len(sz_train), nSzVal=len(sz_val), nSzTest=len(sz_test), nSzAll=len(sz), nMg=len(mg),
              nSzTB=int(sz.label.sum()), nMgTB=int(mg.label.sum()), nMask=len(MASK_CACHE),
              nSub=len(mg_sub), nSeeds=len(CFG["SEEDS"]), nEpochs=CFG["EPOCHS"], areaLung=f"{area_mean:.2f}")
# Rendimiento
for mname in CFG["MODELS"]:
    MACROS[f"params{MK[mname]}"] = f"{PARAMS_M[mname]:.1f}"
    MACROS[f"thr{MK[mname]}"] = f"{RUNS[(mname, SEED0)]['thr']:.2f}"
    for sset in ("sz", "mg"):
        st = perf_stats(mname, sset)
        for k, nm in [("acc", "Acc"), ("f1", "Fs"), ("sens", "Sens"), ("spec", "Spec"), ("auc", "Auc")]:
            MACROS[f"perf{MK[mname]}{SK[sset]}{nm}"] = pm(*st[k], d=3 if k == "auc" else 2)
# XAI
for r in XAI_TABLE:
    mk = "Dense" if r["Modelo"] == "DenseNet121" else "Res"
    key = {"Grad-CAM": "Gc", "LIME": "Lime", "Aleatorio": "Rnd", "Grad-CAM (Mont. completo)": "GcMg", "Grad-CAM (Shen. test)": "GcSz"}[r["Método"]]
    for fld, nm, d in [("EIL", "Eil", 3), ("Ganancia", "Gain", 2), ("Borde", "Bord", 2), ("Deletion", "Del", 3)]:
        if not np.isnan(r[fld][0]): MACROS[f"xai{mk}{key}{nm}"] = pm(*r[fld], d=d)
    if not np.isnan(r["Puntería"][0]): MACROS[f"xai{mk}{key}Pt"] = f"{r['Puntería'][0]:.2f}"
for mname in CFG["MODELS"]:
    s = STATS[mname]; mk = MK[mname]
    MACROS[f"eilOk{mk}"] = f"{s['eil_ok']:.3f}"; MACROS[f"eilErr{mk}"] = "--" if np.isnan(s["eil_err"]) else f"{s['eil_err']:.3f}"
    MACROS[f"nOk{mk}"] = s["n_ok"]; MACROS[f"nErr{mk}"] = s["n_err"]
    MACROS[f"pWilc{mk}"] = "--" if np.isnan(s["p_wilcoxon"]) else (f"{s['p_wilcoxon']:.3f}" if s["p_wilcoxon"] >= 0.001 else "<0.001")
    MACROS[f"pMw{mk}"] = "--" if np.isnan(s["p_mw"]) else f"{s['p_mw']:.3f}"

with open(OUT / "macros.tex", "w", encoding="utf-8") as f:
    f.write("% Generado automáticamente por el notebook\n")
    for k, v in MACROS.items(): f.write(f"\\def\\{k}{{{v}}}\n")

# Tabla de rendimiento (LaTeX)
L = ["\\begin{tabular}{llccccc}", "\\toprule", "Modelo & Conjunto & Acc & F1 & Sens & Espec & AUC \\\\", "\\midrule"]
for mname in CFG["MODELS"]:
    for sset in ("sz", "mg"):
        st = perf_stats(mname, sset)
        L.append(f"{MODEL_LABEL[mname]} & {SET_LABEL[sset]} & " + " & ".join(pm(*st[k], d=3 if k == "auc" else 2) for k in ["acc", "f1", "sens", "spec", "auc"]) + " \\\\")
    if mname != CFG["MODELS"][-1]: L.append("\\midrule")
L += ["\\bottomrule", "\\end{tabular}"]
(OUT / "tab_perf.tex").write_text("\n".join(L), encoding="utf-8")

# Tabla XAI (LaTeX)
cell = lambda t, d=2: "--" if np.isnan(t[0]) else pm(*t, d=d)
L = ["\\begin{tabular}{llcccccc}", "\\toprule", "Modelo & Método & $n$ & EIL & Ganancia & Punt. & Borde & Del.$\\downarrow$ \\\\", "\\midrule"]
for i, r in enumerate(XAI_TABLE):
    pt = "--" if np.isnan(r["Puntería"][0]) else f"{r['Puntería'][0]:.2f}"
    L.append(f"{r['Modelo']} & {r['Método']} & {r['n']} & {cell(r['EIL'], 3)} & {cell(r['Ganancia'])} & {pt} & {cell(r['Borde'])} & {cell(r['Deletion'], 3)} \\\\")
    if (i + 1) % 5 == 0 and i + 1 < len(XAI_TABLE): L.append("\\midrule")
L += ["\\bottomrule", "\\end{tabular}"]
(OUT / "tab_xai.tex").write_text("\n".join(L), encoding="utf-8")

# Registro de configuración y versiones + zip
json.dump(dict(cfg={k: v for k, v in CFG.items()}, torch=torch.__version__, torchvision=torchvision.__version__,
               python=platform.python_version(), device=str(device),
               gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
               thresholds={f"{m}_s{s}": RUNS[(m, s)]["thr"] for (m, s) in RUNS},
               best_epochs={f"{m}_s{s}": RUNS[(m, s)]["best_epoch"] for (m, s) in RUNS}),
          open(OUT / "config.json", "w", encoding="utf-8"), indent=1)
sub_df.to_csv(OUT / "xai_subset_scores.csv", index=False); full_df.to_csv(OUT / "xai_full_scores.csv", index=False)
perf_df.to_csv(OUT / "performance.csv", index=False)
PREDICTIONS = []
for (mname, seed), run in RUNS.items():
    for sset, frame in (("sz", sz_test), ("mg", mg)):
        for i, item in frame.iterrows():
            prob = float(run[f"p_{sset}"][i])
            PREDICTIONS.append(dict(model=mname, seed=seed, dataset=sset, name=item["name"],
                                    label=int(item["label"]), probability_tb=prob,
                                    threshold=run["thr"], predicted=int(prob >= run["thr"])))
pd.DataFrame(PREDICTIONS).to_csv(OUT / "predictions.csv", index=False)
(OUT / "training_history.json").write_text(json.dumps({f"{m}_s{s}": RUNS[(m, s)]["hist"] for m, s in RUNS}, indent=2), encoding="utf-8")

zip_path = OUT.parent / f"{OUT.name}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.glob("*")):
        if p.suffix != ".pt": z.write(p, p.name)               # los pesos (.pt) se omiten del zip por tamaño
print("Archivos en results/:", sorted(p.name for p in OUT.glob("*") if p.suffix != ".pt"))
print("Macros exportadas:", len(MACROS))
try:
    from google.colab import files; files.download(str(zip_path))  # descarga automática en Colab
except Exception:
    pass

### Notas y limitaciones (para la discusión del artículo)
* **Tamaño de muestra pequeño** (≈660 + 138 imágenes): las métricas tienen incertidumbre; por eso se reportan 3 semillas y desviaciones estándar.
* Las máscaras de pulmón existen solo en Montgomery, así que EIL/Ganancia/Puntería se miden allí; en Shenzhen solo la métrica de **Borde**.
* Grad-CAM tiene baja resolución espacial (7×7 interpolado) y LIME depende de la segmentación en superpíxeles y del muestreo (varianza): ninguna explicación es una verdad absoluta (ver *sanity checks*, Adebayo et al., 2018).
* Que el mapa caiga sobre el pulmón **no demuestra** que el razonamiento sea clínicamente correcto (el modelo puede usar señales sutiles del contexto dentro del pulmón).
* **Ejercicio académico:** no es un dispositivo clínico ni validado para diagnóstico.

## 8. Extensión: intervención pareada dentro del pulmón
La elegibilidad y la elección de controles se cierran mediante geometría antes de inferir. Se conserva la forma digital, el número de píxeles, el pulmón y el tercio vertical. Los controles son **regiones sin anotación** y pueden contener hallazgos no marcados. La unidad inferencial es la radiografía; las tres semillas se promedian dentro de cada una. `D > 0` indica una mayor caída del logit positivo menos el normal al intervenir la región índice. Es evidencia de dependencia del clasificador bajo este operador, no causalidad clínica.


In [ ]:
MEAN = np.array(IMAGENET_MEAN, dtype=np.float32)
STD = np.array(IMAGENET_STD, dtype=np.float32)
def model_for(name):
    m = build_model(name, pretrained=False)
    return m, target_layer(m, name)

"""Retrospective, geometry-locked matched intrapulmonary lesion perturbation.

Eligibility is computed without loading model outputs. The independent unit is
the radiograph; seeds and control regions are averaged within that unit.
"""

import argparse
import hashlib
import json
import platform
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from scipy import stats
from scipy.integrate import trapezoid
import scipy
import torch
import torch.nn.functional as F
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget


SIZE = 224
MIN_BOX_COVERAGE = 0.50
MIN_PIXELS = 25
EDGE_TOLERANCE_PX = 5.0
MIN_CONTROLS = 3
MAX_CONTROLS = 5
CONTROL_CENTER_SEPARATION_PX = 12
BLUR_SIGMA = 10.0
ALT_BLUR_SIGMA = 5.0
SEEDS = tuple(CFG["SEEDS"])
MODELS = tuple(CFG["MODELS"])


def image_files(root):
    kinds = ("CXR_png", "leftMask", "rightMask")
    found = {}
    for p in Path(root).rglob("*.png"):
        if p.parent.name in kinds and p.name.startswith("MCUCXR"):
            key = (p.parent.name, p.name)
            if key in found:
                raise ValueError(f"Archivo duplicado: {key}")
            found[key] = p
    return found


def read_case(name, files):
    def gray(kind, method):
        return np.asarray(Image.open(files[(kind, name)]).convert("L").resize((SIZE, SIZE), method))
    image = gray("CXR_png", Image.Resampling.BILINEAR).astype(np.float32) / 255.0
    left = gray("leftMask", Image.Resampling.NEAREST) > 0
    right = gray("rightMask", Image.Resampling.NEAREST) > 0
    if np.any(left & right):
        # The released masks occasionally touch after resizing; assignments
        # remain deterministic because the index is selected by greater area.
        right = right & ~left
    return image, (left, right)


def select_regions(rows, lungs):
    """Return index and translated same-shape controls, or an exclusion reason."""
    union_boxes = boxes_to_mask(rows, SIZE)
    candidates = []
    for box_number, (_, row) in enumerate(rows.iterrows()):
        box = boxes_to_mask(pd.DataFrame([row]), SIZE)
        for side, lung in enumerate(lungs):
            index = box & lung
            area = int(index.sum())
            coverage = area / max(1, int(box.sum()))
            if area >= MIN_PIXELS and coverage >= MIN_BOX_COVERAGE:
                candidates.append((-area, side, box_number, index, box, coverage))
    if not candidates:
        return None, "sin_caja_con_cobertura_y_area_minima"
    _, side, box_number, index, box, coverage = min(candidates, key=lambda t: t[:3])
    lung = lungs[side]
    yy, xx = np.where(index)
    y0, y1, x0, x1 = yy.min(), yy.max()+1, xx.min(), xx.max()+1
    template = index[y0:y1, x0:x1].astype(np.uint8)
    th, tw = template.shape
    area = int(template.sum())
    centroid_y, centroid_x = float(yy.mean()), float(xx.mean())
    third = min(2, int(3 * centroid_y / SIZE))
    distance = cv2.distanceTransform(lung.astype(np.uint8), cv2.DIST_L2, 5)
    index_edge = float(np.mean(distance[index]))
    allowed = (lung & ~union_boxes).astype(np.uint8)
    # MatchTemplate counts eligible pixels under the *same digital shape*.
    counts = cv2.matchTemplate(allowed, template, cv2.TM_CCORR)
    positions = np.argwhere(counts >= area - 0.25)
    ranked = []
    local_y, local_x = np.where(template)
    for top, left in positions:
        cy = float(top + local_y.mean())
        cx = float(left + local_x.mean())
        if min(2, int(3 * cy / SIZE)) != third:
            continue
        edge = float(np.mean(distance[top+local_y, left+local_x]))
        edge_gap = abs(edge - index_edge)
        if edge_gap > EDGE_TOLERANCE_PX:
            continue
        # Fixed geometry-only ranking; ties resolve by y then x.
        cost = (edge_gap, abs(cy-centroid_y), abs(cx-centroid_x), int(top), int(left))
        ranked.append((cost, int(top), int(left), cy, cx, edge))
    ranked.sort(key=lambda t: t[0])
    controls = []
    for _, top, left, cy, cx, edge in ranked:
        if any((cy-q[1])**2 + (cx-q[2])**2 < CONTROL_CENTER_SEPARATION_PX**2 for q in controls):
            continue
        mask = np.zeros((SIZE, SIZE), bool)
        mask[top+local_y, left+local_x] = True
        assert int(mask.sum()) == area and not np.any(mask & union_boxes)
        assert np.all(lung[mask])
        controls.append((mask, cy, cx, edge))
        if len(controls) == MAX_CONTROLS:
            break
    if len(controls) < MIN_CONTROLS:
        return None, f"controles_insuficientes_{len(controls)}"
    return dict(index=index, controls=[v[0] for v in controls], side=side,
                box_number=box_number, box_coverage=coverage, index_area=area,
                centroid_y=centroid_y, centroid_x=centroid_x, third=third,
                index_edge_px=index_edge,
                control_edge_px=[v[3] for v in controls],
                control_centroids=[[v[1], v[2]] for v in controls],
                all_boxes=union_boxes, lung=lung), None


def perturb(image, mask, sigma):
    blurred = cv2.GaussianBlur(image, (0, 0), sigmaX=sigma)
    result = image.copy()
    result[mask] = blurred[mask]
    return result


def tensor(images, device):
    rgb = np.repeat(np.stack(images)[:, :, :, None], 3, axis=3).astype(np.float32)
    return torch.from_numpy(((rgb - MEAN) / STD).transpose(0, 3, 1, 2)).to(device)


def cam_iou_top10(heat, lung, index):
    """IoU with the top 10% lung pixels; flat index resolves map ties."""
    positions = np.flatnonzero(lung)
    count = max(1, int(np.ceil(.10 * len(positions))))
    ranked = positions[np.lexsort((positions, -heat.ravel()[positions]))]
    top = np.zeros(heat.size, bool)
    top[ranked[:count]] = True
    top = top.reshape(lung.shape)
    return float((top & index).sum() / (top | index).sum())


def fidelity_curves(model, image, heat, lung, device):
    """Deletion/insertion of ranked lung pixels using one blurred background."""
    fractions = np.linspace(0, .5, 6)
    positions = np.flatnonzero(lung)
    ranked = positions[np.lexsort((positions, -heat.ravel()[positions]))]
    blurred = cv2.GaussianBlur(image, (0, 0), sigmaX=BLUR_SIGMA)
    full_background = image.copy()
    full_background[lung] = blurred[lung]
    deletion, insertion = [], []
    for fraction in fractions:
        chosen = ranked[:int(np.floor(fraction * len(positions)))]
        d = image.copy()
        i = full_background.copy()
        d.ravel()[chosen] = blurred.ravel()[chosen]
        i.ravel()[chosen] = image.ravel()[chosen]
        deletion.append(d)
        insertion.append(i)
    variants = deletion + insertion
    probs = []
    with torch.no_grad():
        for start in range(0, len(variants), 8):
            p = F.softmax(model(tensor(variants[start:start+8], device)).float(), 1)[:, 1]
            probs.extend(p.cpu().numpy().tolist())
    d, i = np.asarray(probs[:len(fractions)]), np.asarray(probs[len(fractions):])
    area = fractions[-1] - fractions[0]
    return dict(deletion_auc=float(trapezoid(d, fractions) / area),
                insertion_auc=float(trapezoid(i, fractions) / area),
                deletion_curve=json.dumps(d.tolist()), insertion_curve=json.dumps(i.tolist()))


def bootstrap_median(values, seed=193, n=5000):
    a = np.asarray(values, float)
    rng = np.random.default_rng(seed)
    b = np.median(a[rng.integers(len(a), size=(n, len(a)))], axis=1)
    return [float(np.median(a)), *map(float, np.quantile(b, [.025, .975]))]


def signed_rank(values):
    a = np.asarray(values, float)
    a = a[np.isfinite(a)]
    nonzero = a[a != 0]
    out = dict(n=int(len(a)), zeros=int((a == 0).sum()), positive=int((a > 0).sum()),
               negative=int((a < 0).sum()), median_ci=bootstrap_median(a) if len(a) else None,
               scipy=scipy.__version__, method="auto", zero_method="wilcox", alternative="two-sided",
               hypothesis="distribucion_de_D_simetrica_alrededor_de_cero")
    if not len(nonzero):
        out.update(statistic=None, p=None, sign_p=None, rank_biserial=None, status="sin_diferencias_no_nulas")
        return out
    test = stats.wilcoxon(a, zero_method="wilcox", correction=False, alternative="two-sided", method="auto")
    ranks = stats.rankdata(abs(nonzero), method="average")
    pos, neg = ranks[nonzero > 0].sum(), ranks[nonzero < 0].sum()
    out.update(statistic=float(test.statistic), p=float(test.pvalue),
               sign_p=float(stats.binomtest(int((nonzero > 0).sum()), len(nonzero), .5).pvalue),
               rank_biserial=float((pos-neg)/(pos+neg)), status="ok")
    return out


def run(data_root, results, annotations_path, geometry_only=False, output=None):
    data_root, results = Path(data_root), Path(results)
    out = Path(output) if output else results
    out.mkdir(parents=True, exist_ok=True)
    splits = pd.read_csv(results / "splits.csv")
    ann = pd.read_csv(annotations_path)
    required = {"patientId", "x_dis", "y_dis", "width_dis", "height_dis", "Labels"}
    if not required <= set(ann) or set(ann.Labels) != {"TB"}:
        raise ValueError("Formato de cajas NLM inesperado")
    names = sorted(set(ann.patientId))
    frame = splits[(splits.split == "external") & (splits.label == 1)]
    if set(names) - set(frame.name):
        raise ValueError("Cajas que no corresponden a positivos externos")
    files = image_files(data_root)
    manifest, cases = [], {}
    for name in names:
        image, lungs = read_case(name, files)
        region, reason = select_regions(ann[ann.patientId == name], lungs)
        row = dict(name=name, label=1, eligible=region is not None, reason=reason or "",
                   n_boxes=int((ann.patientId == name).sum()),
                   n_controls=len(region["controls"]) if region else 0,
                   side=("leftMask" if region["side"] == 0 else "rightMask") if region else "",
                   box_number=region["box_number"] if region else None,
                   box_coverage=region["box_coverage"] if region else None,
                   index_area=region["index_area"] if region else None,
                   index_edge_px=region["index_edge_px"] if region else None,
                   control_edge_px=json.dumps(region["control_edge_px"]) if region else "",
                   control_centroids=json.dumps(region["control_centroids"]) if region else "",
                   third=region["third"] if region else None)
        manifest.append(row)
        if region:
            assert np.array_equal(perturb(image, np.zeros_like(image, bool), BLUR_SIGMA), image)
            cases[name] = (image, region)
    pd.DataFrame(manifest).to_csv(out / "matched_eligibility.csv", index=False)
    protocol = dict(size=SIZE, min_box_coverage=MIN_BOX_COVERAGE, min_pixels=MIN_PIXELS,
                    edge_tolerance_px=EDGE_TOLERANCE_PX, min_controls=MIN_CONTROLS,
                    max_controls=MAX_CONTROLS, center_separation_px=CONTROL_CENTER_SEPARATION_PX,
                    blur_sigma=BLUR_SIGMA, alternative_blur_sigma=ALT_BLUR_SIGMA,
                    selection="largest box intersection with one lung; ties side then CSV order; controls geometry-only",
                    cam_iou_top_fraction=.10, map_tie_break="ascending flat pixel index",
                    fidelity_fractions=[0, .1, .2, .3, .4, .5],
                    fidelity_background="Gaussian sigma 10 inside lung; outside lung original",
                    map_baselines={"uniform": 1, "smoothed_random_repeats": 10,
                                   "random_grid": [7, 7], "fixed_center_sigma_px": .20*SIZE,
                                   "status": "exploratory diagnostic"},
                    n_annotated=len(names), n_eligible=len(cases),
                    annotation_sha256=hashlib.sha256(Path(annotations_path).read_bytes()).hexdigest(),
                    mode="geometry_only" if geometry_only else "full",
                    hardware=torch.cuda.get_device_name(0) if torch.cuda.is_available() else platform.processor())
    (out / "matched_protocol.json").write_text(json.dumps(protocol, indent=2), encoding="utf-8")
    if geometry_only:
        return protocol
    if not cases:
        raise ValueError("No hay radiografías geométricamente analizables")
    # Descriptive map baselines; generated from geometry only, independent of
    # classifier outputs. Their use is diagnostic, not part of the primary test.
    baseline_rows = []
    grid_y, grid_x = np.mgrid[:SIZE, :SIZE]
    center_map = np.exp(-((grid_x-(SIZE-1)/2)**2 + (grid_y-(SIZE-1)/2)**2) /
                        (2*(.20*SIZE)**2))
    for name, (_, region) in cases.items():
        rng = np.random.default_rng(int(hashlib.sha256(name.encode()).hexdigest()[:8], 16))
        maps = [("Uniforme", 0, np.ones((SIZE, SIZE), float)),
                ("Centro fijo", 0, center_map)]
        maps += [("Aleatorio suavizado", j,
                  cv2.resize(rng.random((7, 7), dtype=np.float32), (SIZE, SIZE),
                             interpolation=cv2.INTER_LINEAR)) for j in range(10)]
        for method, repeat, h in maps:
            total = h[region["lung"]].sum()
            index_mass = h[region["index"]].sum() / total
            controls = [h[m].sum() / total for m in region["controls"]]
            baseline_rows.append(dict(name=name, method=method, repeat=repeat,
                                      index_mass=float(index_mass),
                                      control_median_mass=float(np.median(controls)),
                                      mass_D=float(index_mass-np.median(controls)),
                                      index_iou_top10=cam_iou_top10(h, region["lung"], region["index"])))
    pd.DataFrame(baseline_rows).to_csv(out / "matched_map_baselines.csv", index=False)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    rows = []
    example_name = next(iter(cases))
    example_heat = None
    for model_name in MODELS:
        for seed in SEEDS:
            model, layer = model_for(model_name)
            model.load_state_dict(torch.load(results / f"{model_name}_s{seed}.pt", map_location="cpu", weights_only=True))
            model = model.to(device).eval()
            cam = GradCAM(model=model, target_layers=[layer])
            for name, (image, region) in cases.items():
                masks = [region["index"], *region["controls"]]
                for sigma in (BLUR_SIGMA, ALT_BLUR_SIGMA):
                    variants = [image, *[perturb(image, m, sigma) for m in masks]]
                    with torch.no_grad():
                        logits = model(tensor(variants, device)).float()
                        score = (logits[:, 1] - logits[:, 0]).cpu().numpy()
                        probs = F.softmax(logits, 1)[:, 1].cpu().numpy()
                    drops = score[0] - score[1:]
                    prob_drops = probs[0] - probs[1:]
                    x = tensor([image], device)
                    heat = np.asarray(cam(input_tensor=x, targets=[ClassifierOutputTarget(1)])[0], float)
                    if model_name == MODELS[0] and seed == SEEDS[0] and name == example_name and sigma == BLUR_SIGMA:
                        example_heat = heat.copy()
                    valid_map = bool(np.isfinite(heat).all() and heat[region["lung"]].sum() > 0)
                    if valid_map:
                        h = np.clip(heat, 0, None)
                        lung_mass = h[region["lung"]].sum()
                        index_mass = h[masks[0]].sum() / lung_mass
                        control_mass = [h[m].sum() / lung_mass for m in masks[1:]]
                        iou10 = cam_iou_top10(h, region["lung"], region["index"])
                        fidelity = fidelity_curves(model, image, h, region["lung"], device) if sigma == BLUR_SIGMA else {}
                    else:
                        index_mass, control_mass = np.nan, [np.nan] * (len(masks)-1)
                        iou10, fidelity = np.nan, {}
                    rows.append(dict(model=model_name, seed=seed, name=name, sigma=sigma,
                                     original_logit=float(score[0]), original_probability=float(probs[0]),
                                     index_drop=float(drops[0]), control_drops=json.dumps(drops[1:].tolist()),
                                     D=float(drops[0]-np.median(drops[1:])),
                                     index_probability_drop=float(prob_drops[0]),
                                     control_probability_drops=json.dumps(prob_drops[1:].tolist()),
                                     probability_D=float(prob_drops[0]-np.median(prob_drops[1:])),
                                     valid_map=valid_map, index_cam_mass=float(index_mass),
                                     control_cam_mass=json.dumps(control_mass),
                                     cam_mass_D=float(index_mass-np.median(control_mass)),
                                     cam_iou_top10=float(iou10), **fidelity))
            cam.activations_and_grads.release()
            del cam, model
            if device == "cuda": torch.cuda.empty_cache()
    scores = pd.DataFrame(rows)
    scores.to_csv(out / "matched_scores.csv", index=False)
    per_image = scores.groupby(["model", "sigma", "name"], as_index=False).agg(
        D=("D", "mean"), probability_D=("probability_D", "mean"),
        cam_mass_D=("cam_mass_D", "mean"), valid_maps=("valid_map", "sum"))
    per_image.to_csv(out / "matched_per_image.csv", index=False)
    summary = dict(protocol=protocol, models={})
    for model_name in MODELS:
        summary["models"][model_name] = {}
        for sigma in (BLUR_SIGMA, ALT_BLUR_SIGMA):
            part = per_image[(per_image.model == model_name) & (per_image.sigma == sigma)]
            result = signed_rank(part.D)
            result["n_excluded_geometry"] = len(names) - len(part)
            result["probability_median_ci"] = bootstrap_median(part.probability_D)
            rho_data = part[["D", "cam_mass_D"]].dropna()
            if len(rho_data) >= 4 and rho_data.D.nunique() > 1 and rho_data.cam_mass_D.nunique() > 1:
                result["spearman_rho"] = float(stats.spearmanr(rho_data.D, rho_data.cam_mass_D).statistic)
                rng = np.random.default_rng(992)
                boot = []
                a = rho_data.to_numpy()
                for _ in range(3000):
                    sample = a[rng.integers(len(a), size=len(a))]
                    if len(np.unique(sample[:, 0])) > 1 and len(np.unique(sample[:, 1])) > 1:
                        boot.append(stats.spearmanr(sample[:, 0], sample[:, 1]).statistic)
                result["spearman_ci"] = list(map(float, np.quantile(boot, [.025, .975])))
            else:
                result["spearman_rho"] = result["spearman_ci"] = None
            summary["models"][model_name][str(sigma)] = result
    # ResNet50 is the prespecified secondary model contrast; Holm also covers
    # the alternative intensity as a sensitivity family, separate from primary.
    secondary = [(m, s) for m, s in (("resnet50", BLUR_SIGMA), ("densenet121", ALT_BLUR_SIGMA), ("resnet50", ALT_BLUR_SIGMA))]
    pvals = [summary["models"][m][str(s)]["p"] for m, s in secondary]
    order = np.argsort(pvals)
    adjusted = np.empty(len(pvals))
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (len(pvals)-rank) * pvals[idx])
        adjusted[idx] = min(1.0, running)
    for (m, s), q in zip(secondary, adjusted):
        summary["models"][m][str(s)]["holm_secondary_p"] = float(q)
    (out / "matched_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    table = [r"\begin{tabular}{lccccc}", r"\toprule",
             r"Modelo & $n$ & Mediana $D$ [IC] & $W$ & $p$ & $r_{rb}$\\", r"\midrule"]
    for model_name, label in (("densenet121", "DenseNet121"), ("resnet50", "ResNet50")):
        item = summary["models"][model_name][str(BLUR_SIGMA)]
        median, lo, hi = item["median_ci"]
        table.append(f"{label} & {item['n']} & {median:.3f} [{lo:.3f},{hi:.3f}] & "
                     f"{item['statistic']:.0f} & {item['p']:.5f} & {item['rank_biserial']:.3f}" + r"\\")
    table += [r"\bottomrule", r"\end{tabular}"]
    (out / "tab_matched.tex").write_text("\n".join(table), encoding="utf-8")
    fig, axs = plt.subplots(1, 2, figsize=(8, 3.3), sharey=True)
    for ax, m in zip(axs, MODELS):
        p = per_image[(per_image.model == m) & (per_image.sigma == BLUR_SIGMA)].sort_values("D")
        ax.plot(np.arange(1, len(p)+1), p.D, ".", color="tab:blue" if m == MODELS[0] else "tab:orange")
        ax.axhline(0, color="black", lw=.8)
        ax.set(title=m, xlabel="Radiografías ordenadas por D")
        ax.grid(alpha=.2)
    axs[0].set_ylabel("D: diferencia pareada de logit")
    fig.tight_layout(); fig.savefig(out / "fig_matched_d.png", dpi=220); plt.close(fig)
    image, region = cases[example_name]
    fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
    for ax, with_cam in zip(axes, (False, True)):
        ax.imshow(image, cmap="gray")
        if with_cam and example_heat is not None:
            ax.imshow(example_heat, cmap="jet", alpha=np.clip(example_heat, 0, 1) * .5, vmin=0, vmax=1)
        ax.contour(region["lung"], levels=[.5], colors="yellow", linewidths=.6)
        ax.contour(region["index"], levels=[.5], colors="red", linewidths=1)
        for mask in region["controls"]: ax.contour(mask, levels=[.5], colors="cyan", linewidths=.6)
        ax.set(title="Geometría" if not with_cam else "Grad-CAM positivo, semilla 0")
        ax.axis("off")
    fig.tight_layout(); fig.savefig(out / "fig_matched_example.png", dpi=220); plt.close(fig)
    return summary




In [ ]:
# La prueba rápida verifica adquisición y elegibilidad; la corrida completa infiere con los seis checkpoints.
MATCHED = run(ROOT, OUT, DATA_DIR / 'montgomery_consensus_roi.csv', geometry_only=QUICK)
print(json.dumps(MATCHED, indent=2, ensure_ascii=False))


In [ ]:
# Sensibilidades geométricas posteriores: nunca sustituyen el contraste principal.
if not QUICK:
    main_edge, main_separation = EDGE_TOLERANCE_PX, CONTROL_CENTER_SEPARATION_PX
    sensitivity_rows = []
    for tag, edge, separation, folder in [('principal', 5, 12, OUT),
                                           ('borde_10', 10, 12, OUT / 'edge10_separation12'),
                                           ('borde_10_centros_6', 10, 6, OUT / 'relaxed_geometry')]:
        if tag == 'principal': summary = MATCHED
        else:
            EDGE_TOLERANCE_PX, CONTROL_CENTER_SEPARATION_PX = edge, separation
            summary = run(ROOT, OUT, DATA_DIR / 'montgomery_consensus_roi.csv', output=folder)
        for model_name in MODELS:
            item = summary['models'][model_name][str(BLUR_SIGMA)]
            sensitivity_rows.append(dict(protocol=tag, status='principal' if tag == 'principal' else 'sensibilidad_poshoc',
              model=model_name, n=item['n'], median_D=item['median_ci'][0], ci_low=item['median_ci'][1],
              ci_high=item['median_ci'][2], wilcoxon_p=item['p'], sign_p=item['sign_p'],
              edge_tolerance_px=edge, center_separation_px=separation))
    EDGE_TOLERANCE_PX, CONTROL_CENTER_SEPARATION_PX = main_edge, main_separation
    pd.DataFrame(sensitivity_rows).to_csv(OUT / 'geometry_sensitivity.csv', index=False)
    print(pd.DataFrame(sensitivity_rows).to_string(index=False))


## 9. Ejemplos de pulmones con Grad-CAM y LIME
Las cuatro primeras radiografías elegibles por identificador se muestran con la misma clase positiva y semilla 0 en ambas arquitecturas. Estas láminas son cualitativas y no alteran el contraste principal.


In [ ]:
"""Same-case, positive-class Grad-CAM and LIME examples for the IEEE article."""

import argparse
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lime import lime_image
from lime.wrappers.scikit_image import SegmentationAlgorithm
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import torch
import torch.nn.functional as F


LIME_SAMPLES = 800
LIME_SEED = 271
N_CASES = 4


def lime_positive(model, image, device):
    """Positive LIME coefficients on SLIC superpixels, using the same input scale."""
    rgb = np.repeat(np.round(image[:, :, None] * 255).astype(np.uint8), 3, axis=2)
    segmenter = SegmentationAlgorithm("slic", n_segments=60, compactness=10,
                                      sigma=1, start_label=1)

    @torch.no_grad()
    def predict_fn(batch):
        images = np.asarray(batch, dtype=np.float32) / 255.0
        x = torch.from_numpy(((images - MEAN) / STD).transpose(0, 3, 1, 2)).to(device)
        return F.softmax(model(x).float(), 1).cpu().numpy()

    explainer = lime_image.LimeImageExplainer(random_state=LIME_SEED)
    kwargs = dict(top_labels=2, hide_color=0, num_samples=LIME_SAMPLES,
                  batch_size=8, segmentation_fn=segmenter)
    try:
        explanation = explainer.explain_instance(rgb, predict_fn, progress_bar=False, **kwargs)
    except TypeError:
        explanation = explainer.explain_instance(rgb, predict_fn, **kwargs)
    coefficients = dict(explanation.local_exp[1])
    heat = np.zeros((SIZE, SIZE), dtype=np.float32)
    for segment in np.unique(explanation.segments):
        heat[explanation.segments == segment] = max(0.0, coefficients.get(int(segment), 0.0))
    return heat / heat.max() if heat.max() > 0 else heat


def draw_panel(ax, image, heat, lung, index, controls, title, draw_controls=False):
    ax.imshow(image, cmap="gray", vmin=0, vmax=1)
    if heat is not None:
        ax.imshow(heat, cmap="jet", vmin=0, vmax=1,
                  alpha=np.clip(heat, 0, 1) * .55)
    ax.contour(lung, levels=[.5], colors="yellow", linewidths=.65)
    ax.contour(index, levels=[.5], colors="red", linewidths=.9)
    if draw_controls:
        for mask in controls:
            ax.contour(mask, levels=[.5], colors="cyan", linewidths=.5)
    ax.set_title(title, fontsize=9)
    ax.axis("off")


def run_qualitative(data_root, results, annotations):
    results = Path(results)
    eligible = pd.read_csv(results / "matched_eligibility.csv")
    names = sorted(eligible.loc[eligible.eligible, "name"])[:N_CASES]
    if len(names) != N_CASES:
        raise ValueError("No hay cuatro casos elegibles para ilustrar")
    annotations = pd.read_csv(annotations)
    files = image_files(data_root)
    cases = {}
    for name in names:
        image, lungs = read_case(name, files)
        region, reason = select_regions(annotations[annotations.patientId == name], lungs)
        if reason:
            raise ValueError(f"Cambió la elegibilidad de {name}: {reason}")
        cases[name] = image, region
    preds = pd.read_csv(results / "predictions.csv")
    cfg = json.loads((results / "config.json").read_text(encoding="utf-8"))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    map_arrays = {}
    records = []
    for model_name in ("densenet121", "resnet50"):
        model, layer = model_for(model_name)
        model.load_state_dict(torch.load(results / f"{model_name}_s0.pt", map_location="cpu", weights_only=True))
        model = model.to(device).eval()
        cam = GradCAM(model=model, target_layers=[layer])
        fig, axes = plt.subplots(N_CASES, 3, figsize=(10.2, 12.0))
        for row, name in enumerate(names):
            image, region = cases[name]
            with torch.no_grad():
                probability = float(F.softmax(model(tensor([image], device)).float(), 1)[0, 1].cpu())
            threshold = float(cfg["thresholds"][f"{model_name}_s0"])
            predicted = int(probability >= threshold)
            gradcam = np.asarray(cam(input_tensor=tensor([image], device),
                                     targets=[ClassifierOutputTarget(1)])[0], dtype=np.float32)
            lime = lime_positive(model, image, device)
            map_arrays[f"{model_name}_{name}_gradcam"] = gradcam
            map_arrays[f"{model_name}_{name}_lime"] = lime
            records.append(dict(model=model_name, seed=0, name=name, true_label=1,
                                target_class=1, probability_tb=probability, threshold=threshold,
                                predicted=predicted, lime_samples=LIME_SAMPLES,
                                lime_seed=LIME_SEED, gradcam_valid=bool(gradcam.sum() > 0),
                                lime_valid=bool(lime.sum() > 0)))
            draw_panel(axes[row, 0], image, None, region["lung"], region["index"],
                       region["controls"], f"{name}\np(TB)={probability:.3f}; pred.={'TB' if predicted else 'normal'}", True)
            draw_panel(axes[row, 1], image, gradcam, region["lung"], region["index"],
                       region["controls"], "Grad-CAM, clase TB")
            draw_panel(axes[row, 2], image, lime, region["lung"], region["index"],
                       region["controls"], "LIME, clase TB")
        fig.suptitle("DenseNet121" if model_name == "densenet121" else "ResNet50", fontsize=14)
        fig.tight_layout()
        fig.savefig(results / f"fig_lung_xai_{model_name}.png", dpi=230)
        plt.close(fig)
        cam.activations_and_grads.release()
        del cam, model
        if device == "cuda":
            torch.cuda.empty_cache()
    np.savez_compressed(results / "qualitative_lung_maps.npz", **map_arrays)
    table = pd.DataFrame(records)
    table.to_csv(results / "qualitative_lung_cases.csv", index=False)
    return table




In [ ]:
if not QUICK:
    QUALITATIVE_CASES = run_qualitative(ROOT, OUT, DATA_DIR / 'montgomery_consensus_roi.csv')
    display(QUALITATIVE_CASES)


In [ ]:
# Métricas adicionales a partir de las predicciones exportadas.
from sklearn.metrics import average_precision_score, matthews_corrcoef, brier_score_loss
PRED = pd.read_csv(OUT / 'predictions.csv')
extra = []
for (model_name, seed, dataset), part in PRED.groupby(['model','seed','dataset']):
    y, p, pred = part.label.to_numpy(), part.probability_tb.to_numpy(), part.predicted.to_numpy()
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    extra.append(dict(model=model_name, seed=int(seed), dataset=dataset, n=len(part), prevalence=float(y.mean()),
      threshold=float(part.threshold.iloc[0]), ROC_AUC=float(roc_auc_score(y,p)),
      AP=float(average_precision_score(y,p)), F1=float(2*tp/(2*tp+fp+fn)),
      MCC=float(matthews_corrcoef(y,pred)), Brier=float(brier_score_loss(y,p)),
      TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp)))
extra_df = pd.DataFrame(extra)
extra_df.to_csv(OUT / 'performance_extended.csv', index=False)
tex = [r'\begin{tabular}{llcccccc}', r'\toprule',
       r'Modelo & Prueba & AUC & AP & F1 & MCC & Brier & TN/FP/FN/TP\\', r'\midrule']
for model_name in CFG['MODELS']:
    for dataset, label in [('sz','Shenzhen'), ('mg','Montgomery')]:
        item = extra_df[(extra_df.model == model_name) & (extra_df.seed == 0) & (extra_df.dataset == dataset)].iloc[0]
        model_label = 'DenseNet121' if model_name == 'densenet121' else 'ResNet50'
        tex.append(f"{model_label} & {label} & {item.ROC_AUC:.3f} & {item.AP:.3f} & {item.F1:.3f} & "
                   f"{item.MCC:.3f} & {item.Brier:.3f} & {int(item.TN)}/{int(item.FP)}/{int(item.FN)}/{int(item.TP)}" + r'\\')
tex += [r'\bottomrule', r'\end{tabular}']
(OUT / 'tab_perf_extended.tex').write_text('\n'.join(tex), encoding='utf-8')
print(extra_df.to_string(index=False))


In [ ]:
# Empaquetado final; se excluyen pesos por tamaño.
with zipfile.ZipFile(OUT.parent / f'{OUT.name}.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUT.rglob('*')):
        if path.is_file() and path.suffix != '.pt': z.write(path, path.relative_to(OUT).as_posix())
print('Exportación:', OUT.parent / f'{OUT.name}.zip')
